In [20]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import math

#import xlrd
import openpyxl
import pickle

# import torch
from sklearn.preprocessing import StandardScaler
from joblib import dump,load
from sklearn import utils,preprocessing
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso,LassoCV
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from tensorflow.keras import models
from tensorflow.keras import layers
from keras.models import Sequential
from keras.layers import Dense
#from keras.wrappers.scikit_learn import KerasRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
import datetime
import time

In [ ]:
#Model label
global label
#label="PRI"
#label="FUC"
#label="RFE"
label="GP-2D"
#Initial parameter setting
global N
global n
global test_size     # 回归运行轮次和测试集大小
N=1    # runing time for prediction
n=1    # runing time for 10K-CV
test_size=0.20    # the ratio of testing set
#normalization
def scaling(x):
    sc = StandardScaler()
    x = sc.fit_transform(x)
    return x
#read data
#data = pd.read_excel('origin data.xlsx',header=0)    # Primary features
#data = pd.read_excel('CO-result1.xlsx',header=0)    # Primary features using statistical functions
#data = pd.read_excel('list-RFE.xlsx',header=0)    # Primary features using RFE
data = pd.read_excel('EG-1.xlsx',header=0)    # Primary features using Gplearn

x_data = np.array(data.iloc[:,1:])    # features
y_data = np.array(data.iloc[:,0])    # ΔGH
print(y_data)

#The number of Sample
global size
size = x_data.shape[1]
size

In [22]:
##  Model regression
def fitRegression(inputData,outputData,test_size,N,regr):
    r2_train = []    # training R2
    r2_test = []    # testing R2
    rmse_train = []    # training rmse
    rmse_test = []    # testing rmse
    mae_train = []
    mae_test = []
    best_r2 = []
    best_rmse = []
    for i in range(N):
        print('---------round: %d---------'%(i+1))
        X_train,X_test,y_train,y_test = train_test_split(inputData,outputData,test_size=test_size,random_state=None)    # Randomly split data                                                                                                                # 该方法对dataframe也适用
        score_train = r2_score(y_train,regr.predict(pd.DataFrame(X_train)))
        error_train = np.sqrt(sum((y_train-regr.predict(pd.DataFrame(X_train)))**2)/max(y_train.shape))   
        mae0_train = sum(abs(y_train-regr.predict(pd.DataFrame(X_train))))/max(y_train.shape)
        r2_train.append(score_train)   
        rmse_train.append(error_train)
        mae_train.append(mae0_train)
        print('r2_train:   %.3f'%score_train)
        print('RMSE_train: %.3f'%error_train)
        print('MAE_train: %.3f'%mae0_train)
        
        score_test = r2_score(y_test,regr.predict(pd.DataFrame(X_test)))
        error_test = np.sqrt(sum((y_test-regr.predict(pd.DataFrame(X_test)))**2)/max(y_test.shape))
        mae0_test = sum(abs(y_test-regr.predict(pd.DataFrame(X_test))))/max(y_test.shape)
        r2_test.append(score_test)   
        rmse_test.append(error_test)
        mae_test.append(mae0_test)
        print('r2_test:   %.3f'%score_test)
        print('RMSE_test: %.3f'%error_test)
        print('MAE_test: %.3f'%mae0_test)
        if i == 0:
            max_score = score_test           #  At run 0, record R2 and the model at this point
            worst_mae = mae_test
            y_train_best = y_train
            y_test_best = y_test
            y_train_predict_best = regr.predict(pd.DataFrame(X_train))
            y_test_predict_best = regr.predict(pd.DataFrame(X_test))
            best_r2 = score_test
            best_rmse = error_test
        if i>0:
            if score_test>max_score:
                max_score = score_test       #  Update the maximum R2 value and save the model and the data
                y_train_best = y_train
                y_test_best = y_test
                y_train_predict_best = regr.predict(pd.DataFrame(X_train))
                y_test_predict_best = regr.predict(pd.DataFrame(X_test))
                best_r2 = score_test
                best_rmse = error_test
            if mae_test>worst_mae:
                worst_mae = mae_test
        print('--------------------------') 
    return r2_train,rmse_train,mae_train,r2_test,rmse_test,mae_test,best_r2,best_rmse,y_train_best,y_train_predict_best,y_test_best,y_test_predict_best,worst_mae

In [ ]:
##  Hyperparameter optimization
def optHyperparameter(inputData,outputData,test_size,n,search):
    r2 = []
    for i in range(n):
        X_train,X_test,y_train,y_test = train_test_split(inputData,outputData,test_size=test_size,random_state=None)    
        search.fit(X_train,y_train)
        regr = search.best_estimator_
        r2 = r2_score(y_train,regr.predict(X_train))
        if i == 0:
            max_score = r2          
            best_regr = regr    
        if i>0:
            if r2>max_score:
                max_score = r2       
                best_regr = regr  
    return max_score,best_regr
##  GridCv generation
def generateSearch(model,method,param_grid,n_jobs,cv):    
    if model=='gbr':
        gbr = GradientBoostingRegressor()
        if method=='grid_search':
            search = GridSearchCV(gbr,param_grid=param_grid,n_jobs=n_jobs,cv=cv,scoring = 'neg_mean_squared_error')
            return search
        elif method=='randomized_search':
            search = RandomizedSearchCV(gbr,param_distributions=param_grid,n_jobs=n_jobs,cv=cv,scoring = 'neg_mean_squared_error')
            return search     

#GBR
def generateGbrParamGrid(low1,high1,num1,low2,high2,num2,low3,high3,num3,subsample):      # 
    param_grid = {
        'n_estimators':np.arange(int(low1),int(high1),int(num1)),
        'max_depth':np.arange(int(low2),int(high2),int(num2)),
        'learning_rate':np.arange((low3),(high3),(num3)),
        'subsample':[subsample],




    }
    return param_grid
param = generateGbrParamGrid(50,100,20, 10,15,1,  0.1,0.9,0.2, 1)
param
search = generateSearch('gbr','grid_search',param,-1,10)
search
max_score,best_regr = optHyperparameter(x_data,y_data,test_size,n,search)
max_score
best_regr
r2_train,rmse_train,mae_train,r2_test,rmse_test,mae_test,best_r2,best_rmse,y_train_best,y_train_predict_best,y_test_best,y_test_predict_best,xgbr_worst_mae= \
fitRegression(x_data,y_data,test_size,N,best_regr)
#plotBar(r2_train,rmse_train,r2_test,rmse_test,N,'GBR',label,'saving')
#plotScatter(best_r2,best_rmse,y_train_best,y_train_predict_best,y_test_best,y_test_predict_best,'GBR',label,'saving')
best_gbr = best_regr
dump(best_gbr, "Models-gbr_"+label+".joblib")    # 存储至硬盘
rmse_r2 = [rmse_train, rmse_test, r2_train, r2_test]
mean_rmse_r2 = np.mean(rmse_r2,1) 
std_err = np.std(rmse_r2,axis=1,ddof=1)/np.sqrt(N) 
gbr_predicted_result = [mean_rmse_r2,std_err]
gbr_predicted_result = pd.DataFrame(gbr_predicted_result)
gbr_predicted_result.columns = ["Training rmse", "Testing rmse", "Training R2", "Testing R2"]
gbr_predicted_result.index = ["Mean", "std"]
gbr_predicted_result.to_csv("GBR_"+label+".csv",sep=',')
gbr_mae = [mae_train, mae_test]
gbr_mae = pd.DataFrame(gbr_mae)
gbr_mae.to_csv("GBR_MAE.csv",sep=',')
#read data
rawData1 = pd.read_excel('pred.xlsx')
inputData1 = rawData1.iloc[:,1:]
inputData1 = scaling(inputData1)
#load GP
best_gp = pickle.load(open("2Dgp_2D.pickle.dat", "rb"))
gp_features_= best_gp.transform(inputData1)
#transform features of 126 MAZ
data1 =pd.DataFrame(gp_features_)
data1.to_csv("HCOOH-predict-GP-2D.csv",index=False,sep=',')
#load xgbr
best_regr = load("Models-gbr_GP-2D.joblib")
best_regr
#prediction

Y_predict = best_regr.predict(data1.values)
Y_predict =pd.DataFrame(Y_predict)
Y_predict
#output result
Y_predict.to_csv("HCOOH-predict-GP.csv",index=False,sep=',')